# Chapter 4: Neural Networks and Statistical Mechanics
## Practical: Perceptron and Hopfield Network

## Part 1: Perceptron for Linearly Separable Data

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris
from sklearn.linear_model import Perceptron
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

# Load iris: use only two classes and first two features for 2D plotting
iris = load_iris()
X = iris.data[iris.target != 2][:, :2]
y = iris.target[iris.target != 2]
y = np.where(y == 0, -1, 1)  # convert to -1, +1

# Split and standardise
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Perceptron
clf = Perceptron(max_iter=1000, tol=1e-3, random_state=42)
clf.fit(X_train_scaled, y_train)
y_pred = clf.predict(X_test_scaled)

print(f"Accuracy: {accuracy_score(y_test, y_pred):.3f}")
print(f"Number of iterations: {clf.n_iter_}")

# Visualise decision boundary
xx, yy = np.meshgrid(np.linspace(-3, 3, 50), np.linspace(-3, 3, 50))
X_plot = np.c_[xx.ravel(), yy.ravel()]
Z = clf.predict(X_plot).reshape(xx.shape)

plt.figure(figsize=(8, 6))
plt.contourf(xx, yy, Z, alpha=0.3, cmap='coolwarm')
plt.scatter(X_train_scaled[:, 0], X_train_scaled[:, 1], c=y_train, cmap='coolwarm', 
            edgecolors='k', s=50)
plt.title('Perceptron decision boundary (first two features)')
plt.xlabel('Sepal length (scaled)')
plt.ylabel('Sepal width (scaled)')
plt.show()

## Part 2: Hopfield Network for Pattern Completion

In [ ]:
def hopfield_update(s, W):
    """Asynchronous update: update neurons one by one."""
    N = len(s)
    order = np.random.permutation(N)
    for i in order:
        field = np.dot(W[i], s)
        s[i] = 1 if field >= 0 else -1
    return s

def energy(s, W):
    return -0.5 * np.dot(s, np.dot(W, s))

def pattern_to_vector(P):
    return P.flatten()

# Create patterns: 'L', 'T', 'F' (5x3)
L = np.array([
    [1, -1, -1],
    [1, -1, -1],
    [1, -1, -1],
    [1, -1, -1],
    [1, 1, 1]
])

T = np.array([
    [1, 1, 1],
    [-1, 1, -1],
    [-1, 1, -1],
    [-1, 1, -1],
    [-1, 1, -1]
])

F = np.array([
    [1, 1, 1],
    [1, -1, -1],
    [1, 1, 1],
    [1, -1, -1],
    [1, -1, -1]
])

patterns = [L, T, F]
patterns_vec = [pattern_to_vector(p) for p in patterns]
N = patterns_vec[0].shape[0]

# Hebbian learning
W = np.zeros((N, N))
for mu in range(len(patterns_vec)):
    xi = patterns_vec[mu]
    W += np.outer(xi, xi) / N
np.fill_diagonal(W, 0)  # no self-connections

# Test recall from noisy input (flip 30% of bits)
test_vec = patterns_vec[1].copy()
noise_idx = np.random.choice(N, size=int(0.3*N), replace=False)
test_vec[noise_idx] *= -1

# Recover
s = test_vec.copy()
energy_history = []
for step in range(20):
    energy_history.append(energy(s, W))
    s = hopfield_update(s, W)

print(f"Final energy: {energy_history[-1]:.2f}")
print(f"Stored pattern energy: {energy(patterns_vec[1], W):.2f}")

# Plot original, noisy, and recovered
fig, axes = plt.subplots(1, 3, figsize=(10, 3))
axes[0].imshow(patterns_vec[1].reshape(5, 3), cmap='binary', interpolation='nearest')
axes[0].set_title('Original')
axes[0].axis('off')

axes[1].imshow(test_vec.reshape(5, 3), cmap='binary', interpolation='nearest')
axes[1].set_title('Noisy input (30% flips)')
axes[1].axis('off')

axes[2].imshow(s.reshape(5, 3), cmap='binary', interpolation='nearest')
axes[2].set_title('Recovered')
axes[2].axis('off')
plt.tight_layout()
plt.show()

# Plot energy relaxation
plt.figure(figsize=(8, 5))
plt.plot(energy_history, 'o-', linewidth=2, markersize=8)
plt.xlabel('Update step', fontsize=12)
plt.ylabel('Energy', fontsize=12)
plt.title('Energy minimisation during recall', fontsize=14)
plt.grid(True, alpha=0.3)
plt.show()

### Observations

- The perceptron converges to a linear decision boundary.
- The Hopfield network stores patterns as energy minima.
- Noisy input is attracted to the nearest stored pattern.
- Energy decreases monotonically with each update.
- Capacity is limited (P_max ≈ 0.138N) as predicted by spin glass theory.